In [ ]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [2]:
df_raw = pd.read_csv('/Users/hariz/Desktop/TMDB-movie-analysis/extract_movie_features.csv')


In [3]:
df = df_raw.copy()
df.columns

Index(['movie_id', 'imdb_id', 'title', 'original_title', 'status',
       'original_language', 'adult', 'video', 'release_date', 'release_year',
       'release_month', 'release_day_of_week', 'budget', 'revenue',
       'net_profit', 'roi', 'runtime', 'movie_popularity', 'vote_average',
       'vote_count', 'collection_name', 'is_part_of_franchise',
       'primary_genre', 'primary_production_company', 'primary_country',
       'director_name', 'cast', 'created_at', 'updated_at'],
      dtype='str')

In [4]:
df_feature = df[['original_language', 'release_year', 'release_month', 'release_day_of_week', 'budget', 'revenue', 'runtime', 'collection_name', 'is_part_of_franchise', 'primary_genre', 'primary_production_company', 'primary_country', 'director_name', 'cast']].copy()

df_feature.sort_values(by='collection_name', ascending=False).head(4)


,original_language,release_year,release_month,release_day_of_week,budget,revenue,runtime,collection_name,is_part_of_franchise,primary_genre,primary_production_company,primary_country,director_name,cast
7777,zh,2025.0,6.0,6.0,40000000,52327026,96,酱园弄,1,Drama,China Film Group Corporation,China,Peter Chan Ho-Sun,"Yin Fang, Ci Sha, Lin Yongjian, Zhao Liying, Z..."
6743,ru,2017.0,8.0,4.0,686410,6143216,81,Бабушка лёгкого поведения,1,Comedy,Vice Films,Russia,Maryus Vaysberg,"Filipp Kirkorov, Elena Valyushkina, Vladimir S..."
6643,tr,2017.0,1.0,5.0,3912363,9072414,116,Çalgı Çengi Collection,1,Comedy,TR 40 33 Production,Turkey,Selçuk Aydemir,"Rasim Öztekin, Cahit Gök, Korhan Herduran, Bur..."
2782,en,2005.0,4.0,3.0,60000000,71410636,101,xXx Collection,1,Adventure,Original Film,United States,Lee Tamahori,"Scott Speedman, Peter Strauss, Jeanne Sakata, ..."


In [5]:
df_feature['primary_genre'].nunique()


17

In [6]:
df_feature.info()

<class 'pandas.DataFrame'>
RangeIndex: 8040 entries, 0 to 8039
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   original_language           8040 non-null   str    
 1   release_year                8040 non-null   float64
 2   release_month               8040 non-null   float64
 3   release_day_of_week         8040 non-null   float64
 4   budget                      8040 non-null   int64  
 5   revenue                     8040 non-null   int64  
 6   runtime                     8040 non-null   int64  
 7   collection_name             2197 non-null   str    
 8   is_part_of_franchise        8040 non-null   int64  
 9   primary_genre               8039 non-null   str    
 10  primary_production_company  7983 non-null   str    
 11  primary_country             7972 non-null   str    
 12  director_name               8035 non-null   str    
 13  cast                        8038 non-null   

In [7]:
# Convert raw budget and revenue to millions while keeping them numeric
df_feature['budget_m'] = df_feature['budget'] / 1e6
df_feature['revenue_m'] = df_feature['revenue'] / 1e6
df_feature['roi_ratio'] = df_feature['revenue'] / df['budget']
df_feature['log_budget'] = np.log1p(df_feature['budget'])
df_feature['log_revenue'] = np.log1p(df_feature['revenue'])

df_feature.describe()

,release_year,release_month,release_day_of_week,budget,revenue,runtime,is_part_of_franchise,budget_m,revenue_m,roi_ratio,log_budget,log_revenue
count,8040.000000,8040.000000,8040.000000,8.040000e+03,8.040000e+03,8040.000000,8040.000000,8040.000000,8040.000000,8040.000000,8040.000000,8040.000000
mean,2005.207587,6.936567,4.015050,3.227807e+07,9.374970e+07,112.687562,0.273259,32.278066,93.749701,4.252037,16.597706,17.325022
std,14.151327,3.443638,1.254085,4.309790e+07,1.826120e+08,24.359273,0.445660,43.097903,182.611958,11.710634,1.246018,1.442861
min,1925.000000,1.000000,0.000000,5.017170e+05,9.626200e+04,31.000000,0.000000,0.501717,0.096262,0.100000,13.125793,11.474839
25%,1997.000000,4.000000,3.000000,7.000000e+06,1.129993e+07,97.000000,0.000000,7.000000,11.299933,1.000000,15.761421,16.240307
50%,2008.000000,7.000000,4.000000,1.800000e+07,3.114227e+07,108.000000,0.000000,18.000000,31.142267,2.128923,16.705882,17.254076
75%,2016.000000,10.000000,5.000000,3.850000e+07,9.416589e+07,123.000000,1.000000,38.500000,94.165891,4.209437,17.466169,18.360569
max,2027.000000,12.000000,6.000000,6.588000e+08,2.923706e+09,622.000000,1.000000,658.800000,2923.706026,668.884953,20.305931,21.796118


In [12]:
def evaluate_model(df, feature_list, categorical_cols, model_name="Model"):
    """
    Trains a Random Forest model on log_revenue and evaluates in dollar metrics.
    """
    X = df[feature_list].copy()
    numeric_cols = [c for c in feature_list if c not in categorical_cols]
    
    y_log = df['log_revenue'] if 'log_revenue' in df.columns else np.log1p(df['revenue'])
    y_dollar = df['revenue']

    # Preprocess missing values
    X[numeric_cols] = X[numeric_cols].fillna(0)
    X[categorical_cols] = X[categorical_cols].fillna('Unknown').astype(str)
    
    # Train/Test Split (Fixed random_state=42 for direct baseline comparison)
    X_train, X_test, y_train_log, y_test_log, y_train_dollar, y_test_dollar = train_test_split(
        X, y_log, y_dollar, test_size=0.2, random_state=42
    )
    
    # Ordinal Encode Categoricals
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
    X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])
    
    # Train
    model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train_log)
    
    # Predict and Inverse Transform back to Dollars
    log_preds = model.predict(X_test)
    dollar_preds = np.clip(np.expm1(log_preds), a_min=0, a_max=None)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test_dollar, dollar_preds))
    mae = mean_absolute_error(y_test_dollar, dollar_preds)
    r2 = r2_score(y_test_dollar, dollar_preds)
    
    print(f"\n================ {model_name} ================")
    print(f"Dollar RMSE: ${rmse:,.2f}")
    print(f"Dollar MAE:  ${mae:,.2f}")
    print(f"Dollar R²:   {r2:.4f}")
    
    return {'model_name': model_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}